<h1>RAG - Retrieval Augmented Generation</h1>

**What you'll learn:**
- Why LLMs need RAG (the core problem)
- Typical RAG Pipeline
- Building a complete working RAG system from scratch

## Section 1: The Problem with LLMs

### The 3 Core Problems RAG Solves

| Problem | What Happens Without RAG | How RAG Fixes It |
|---------|--------------------------|------------------|
| **Knowledge Cutoff** | LLM can't answer about events after training | Retrieve fresh docs at query time |
| **Hallucination** | LLM confidently invents facts | Ground answers in retrieved evidence |
| **Private Data** | LLM never saw your internal docs | Index your docs, retrieve on demand |

---

### RAG Definition

> **Retrieval-Augmented Generation (RAG)** takes an input query, retrieves a set of relevant supporting documents from an external knowledge source, and feeds both the query AND the retrieved context to an LLM to generate a grounded, accurate response.
>
> *— Lewis et al., 2020 (original RAG paper)*

RAG is particularly useful in **knowledge-intensive scenarios** or **domain-specific applications** that require knowledge that's continually updating.

## Section 2: Typical RAG Pipeline

 ![title](RagFlow.png)

## Section 3: Setup

In [1]:
# Install required packages
%pip install sentence-transformers numpy scikit-learn matplotlib --quiet
print("✅ Packages installed!")

Note: you may need to restart the kernel to use updated packages.
✅ Packages installed!


## Section 4: Build Our Knowledge Base

We'll build a **Company HR Assistant** a chatbot that answers questions from a company's HR handbook. A regular LLM doesn't know your company's specific policies, but RAG can fix that.

In [2]:
# ============================================================
# 📚 KNOWLEDGE BASE: Abcd Corp HR Handbook
# (In real life, these would come from PDFs, Word docs, etc.)
# ============================================================

hr_documents = [
    """
    VACATION & TIME OFF POLICY
    
    Full-time employees accrue 15 days of paid vacation per year for the first 3 years.
    After 3 years of service, employees accrue 20 days per year.
    Senior employees (7+ years) accrue 25 days per year.
    
    Vacation requests must be submitted at least 2 weeks in advance through the HR portal.
    Maximum 10 consecutive vacation days can be taken without VP approval.
    Unused vacation rolls over up to a maximum of 30 days.
    Employees leaving the company are paid out unused vacation days.
    """,

    """
    REMOTE WORK POLICY
    
    Abcd Corp operates a hybrid work model.
    Employees are required to be in office on Tuesdays and Thursdays (core days).
    Monday, Wednesday, and Friday are flexible remote days.
    
    Remote work from international locations requires approval from your manager and HR.
    The maximum international remote work period is 30 days per year.
    All employees must be reachable during their local business hours (9am-6pm).
    Equipment: Abcd provides a $1,500 home office stipend for approved remote setups.
    """,

    """
    HEALTH BENEFITS & INSURANCE
    
    Abcd Corp offers three health plan tiers: Basic, Standard, and Premium.
    
    Basic Plan: Company pays 70%, employee pays 30% of monthly premium.
    Standard Plan: Company pays 80%, employee pays 20%. Includes dental and vision.
    Premium Plan: Company pays 90%, employee pays 10%. Includes dental, vision, and mental health.
    
    Enrollment window is 30 days after hire date. Annual open enrollment is in November.
    Dependents can be added during enrollment. Domestic partners are eligible.
    FSA (Flexible Spending Account) available: up to $2,850 per year pre-tax.
    """,

    """
    PERFORMANCE REVIEW & SALARY REVIEW PROCESS
    
    Performance reviews are conducted twice per year: June and December.
    The review cycle includes self-evaluation, peer feedback, and manager review.
    Employees are rated on a 5-point scale: 1=Needs Improvement, 3=Meets Expectations, 5=Exceptional.
    
    Salary increases are tied to performance ratings:
    - Rating 1-2: No increase, Performance Improvement Plan may be required
    - Rating 3: 2-4% merit increase
    - Rating 4: 4-7% merit increase
    - Rating 5: 7-12% merit increase + eligibility for spot bonuses
    
    Promotions are discussed during the December cycle.
    """,

    """
    PARENTAL LEAVE POLICY
    
    Primary caregivers receive 16 weeks of fully paid parental leave.
    Secondary caregivers receive 6 weeks of fully paid parental leave.
    Leave applies to birth, adoption, and foster placement.
    
    Employees must have been employed for at least 6 months to qualify.
    Leave must be taken within 12 months of the qualifying event.
    Part-time return to work (50% schedule) is available for the first 4 weeks back.
    Baby bonus: $500 gift card provided upon return from parental leave.
    """,

    """
    LEARNING & DEVELOPMENT BUDGET
    
    Each employee receives $2,000 per year for learning and development.
    Approved uses: online courses, conferences, books, certifications, workshops.
    
    Requests must be submitted in advance via the L&D portal.
    Manager approval required for expenses over $500.
    Certification exam fees are covered at 100%.
    If an employee leaves within 6 months of a company-sponsored course, they may be asked to repay the cost.
    Conference travel (flights + hotel) can be expensed separately from the L&D budget.
    """
]

print(f"📚 Knowledge base loaded: {len(hr_documents)} HR policy documents")
for i, doc in enumerate(hr_documents):
    title = doc.strip().split('\n')[0]
    print(f"   {i+1}. {title}")

📚 Knowledge base loaded: 6 HR policy documents
   1. VACATION & TIME OFF POLICY
   2. REMOTE WORK POLICY
   3. HEALTH BENEFITS & INSURANCE
   4. PERFORMANCE REVIEW & SALARY REVIEW PROCESS
   5. PARENTAL LEAVE POLICY
   6. LEARNING & DEVELOPMENT BUDGET


## Section 5: Document Chunking

In real RAG systems, documents are **split into smaller chunks** before embedding. 

In [3]:
# Fixed-size chunking with overlap
def simple_chunker(text, chunk_size=200, overlap=50):
    """
    Simple word-based chunker with overlap.
    In production, use RecursiveCharacterTextSplitter from LangChain
    or similar tools.
    """
    words = text.split()
    chunks = []
    start = 0
    
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap  # overlap for context continuity
        
    return chunks

# For our short HR docs, we'll treat each document as one chunk
# In practice with longer docs, you'd chunk each one
all_chunks = []
for doc in hr_documents:
    # Clean and use doc directly (they're short enough)
    all_chunks.append(doc.strip())

print(f" Total chunks ready for embedding: {len(all_chunks)}")
print(f"\n Chunk size stats:")
for i, chunk in enumerate(all_chunks):
    word_count = len(chunk.split())
    print(f"   Chunk {i+1}: {word_count} words")

 Total chunks ready for embedding: 6

 Chunk size stats:
   Chunk 1: 87 words
   Chunk 2: 77 words
   Chunk 3: 87 words
   Chunk 4: 86 words
   Chunk 5: 79 words
   Chunk 6: 78 words


## Section 6: Embedding: Converting Text to Vectors

In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

print(" Loading embedding model (first run downloads ~80MB)...")
# all-MiniLM-L6-v2 is fast, small, and great for demos
# Production alternatives: text-embedding-ada-002, voyage-3, BGE-large
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print(" Embedding model loaded!")

print("\n Embedding all document chunks...")
chunk_embeddings = embedding_model.encode(all_chunks, show_progress_bar=True)

print(f"\n Embedding complete!")
print(f"   Shape: {chunk_embeddings.shape}")
print(f"   ({len(all_chunks)} chunks × {chunk_embeddings.shape[1]} dimensions each)")
print(f"\n First 5 values of the 'Vacation Policy' embedding:")
print(f"   {chunk_embeddings[0][:5].round(4)}")

 Loading embedding model (first run downloads ~80MB)...
 Embedding model loaded!

 Embedding all document chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


 Embedding complete!
   Shape: (6, 384)
   (6 chunks × 384 dimensions each)

 First 5 values of the 'Vacation Policy' embedding:
   [ 0.0513  0.0358  0.0364 -0.0185 -0.0095]


## Section 7: The Retriever

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(query, top_k=2, threshold=0.0):
    """
    Core retrieval function:
    1. Embed the query
    2. Compare to all chunk embeddings via cosine similarity
    3. Return top-k most relevant chunks
    """
    # Embed the query using the SAME model used for documents
    query_embedding = embedding_model.encode([query])
    
    # Calculate cosine similarity between query and all chunks
    similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]
    
    # Get top-k indices sorted by similarity (highest first)
    top_k_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_k_indices:
        if similarities[idx] >= threshold:
            results.append({
                'chunk': all_chunks[idx],
                'score': float(similarities[idx]),
                'index': int(idx)
            })
    
    return results

# ── Test the retriever ──────────────────────────────────────
test_queries = [
    "How many vacation days do I get?",
    "Can I work from another country?",
    "How much money for training courses?"
]

for q in test_queries:
    results = retrieve(q, top_k=1)
    top_doc_title = results[0]['chunk'].strip().split('\n')[0]
    print(f"Query: '{q}'")
    print(f"  → Best match: {top_doc_title} (score: {results[0]['score']:.3f})")
    print()

Query: 'How many vacation days do I get?'
  → Best match: VACATION & TIME OFF POLICY (score: 0.657)

Query: 'Can I work from another country?'
  → Best match: REMOTE WORK POLICY (score: 0.304)

Query: 'How much money for training courses?'
  → Best match: LEARNING & DEVELOPMENT BUDGET (score: 0.628)



## Section 8: Prompt Augmentation

In [7]:
def build_rag_prompt(query, retrieved_docs):
    """
    Build the augmented prompt by combining:
    - System instructions (enforce grounding)
    - Retrieved context
    - User's question
    """
    context_parts = []
    for i, doc in enumerate(retrieved_docs):
        context_parts.append(
            f"[Source {i+1} | Relevance: {doc['score']:.3f}]\n{doc['chunk']}"
        )
    
    context = "\n\n".join(context_parts)
    
    prompt = f"""You are an HR assistant for Abcd Corp.
        Answer questions using ONLY the provided HR policy documents below.
        If the answer is not in the documents, say: "I don't have information about that in the HR policies."
        Always cite which policy section your answer comes from.
        
        === HR POLICY DOCUMENTS ===
        {context}
        === END OF DOCUMENTS ===
        
        Employee Question: {query}
        
        Answer:"""
    
    return prompt

# Show example augmented prompt
sample_query = "How many vacation days do I get after 5 years?"
docs = retrieve(sample_query, top_k=1)
prompt = build_rag_prompt(sample_query, docs)
print("📝 Example Augmented Prompt:")
print("="*60)
print(prompt[:700] + "...")
print("="*60)

📝 Example Augmented Prompt:
You are an HR assistant for Abcd Corp.
        Answer questions using ONLY the provided HR policy documents below.
        If the answer is not in the documents, say: "I don't have information about that in the HR policies."
        Always cite which policy section your answer comes from.
        
        === HR POLICY DOCUMENTS ===
        [Source 1 | Relevance: 0.619]
VACATION & TIME OFF POLICY
    
    Full-time employees accrue 15 days of paid vacation per year for the first 3 years.
    After 3 years of service, employees accrue 20 days per year.
    Senior employees (7+ years) accrue 25 days per year.
    
    Vacation requests must be submitted at least 2 weeks in advance through the ...


## Section 9: Full RAG Pipeline with LLM Generation

In [11]:
import os
import ollama

def rag_pipeline(question, top_k=2, show_steps=True):
    """
    Complete RAG Pipeline:
    Retrieve → Augment → Generate
    """
    if show_steps:
        print(f"\n{'='*65}")
        print(f" QUESTION: {question}")
        print(f"{'='*65}")
    
    # ── STEP 1: RETRIEVE ──────────────────────────────────────
    if show_steps:
        print(f"\n Step 1: Retrieving top-{top_k} relevant documents...")
    
    relevant_docs = retrieve(question, top_k=top_k)
    
    if show_steps:
        for doc in relevant_docs:
            title = doc['chunk'].strip().split('\n')[0]
            print(f"   ✓ [{doc['score']:.3f}] {title}")
    
    # ── STEP 2: AUGMENT ───────────────────────────────────────
    if show_steps:
        print(f"\n Step 2: Building augmented prompt...")
    
    prompt = build_rag_prompt(question, relevant_docs)
    
    if show_steps:
        print(f"   ✓ Prompt ready ({len(prompt)} chars)")
    
    # ── STEP 3: GENERATE ──────────────────────────────────────
    if show_steps:
        print(f"\n Step 3: Generating answer...\n")
    
    response = ollama.chat(
        model='llama3',
        messages=[{"role": "user", "content": prompt}]
    )

    answer = response['message']['content']
    
    if show_steps:
        print(f" HR Assistant Answer:")
        print(f"   {answer}")
        print(f"\n{'='*65}")
    
    return answer

print("RAG pipeline ready!")

RAG pipeline ready!


In [12]:
rag_pipeline("I've been at the company for 5 years. How many vacation days do I get?")


 QUESTION: I've been at the company for 5 years. How many vacation days do I get?

 Step 1: Retrieving top-2 relevant documents...
   ✓ [0.665] VACATION & TIME OFF POLICY
   ✓ [0.363] REMOTE WORK POLICY

 Step 2: Building augmented prompt...
   ✓ Prompt ready (1650 chars)

 Step 3: Generating answer...

 HR Assistant Answer:
   According to our VACATION & TIME OFF POLICY [Source 1], after 3 years of service, employees accrue 20 days per year. Since you have been with the company for 5 years (which exceeds the 3-year mark), you would accrue 20 days per year.

Policy section: VACATION & TIME OFF POLICY



'According to our VACATION & TIME OFF POLICY [Source 1], after 3 years of service, employees accrue 20 days per year. Since you have been with the company for 5 years (which exceeds the 3-year mark), you would accrue 20 days per year.\n\nPolicy section: VACATION & TIME OFF POLICY'

## Section 10: RAG vs. No-RAG Comparison

In [13]:
question = "What is the parental leave policy for secondary caregivers at Abcd Corp?"

print("WITHOUT RAG (plain LLM):")
print("="*65)

plain_response = ollama.chat(
        model='llama3',
        messages=[{"role": "user", "content": f"Answer this: {question}"}]
    )

print(plain_response['message']['content'])

print("\n\n WITH RAG (company policy retrieved):")
print("="*65)
rag_pipeline(question, show_steps=False)

WITHOUT RAG (plain LLM):
I'm just an AI, I don't have access to specific company policies or information. However, I can provide general information about parental leave policies.

Parental leave policies typically vary by company and industry. Some companies may offer more comprehensive policies than others. If you are looking for information on a specific company's policy, I recommend checking the company's website or contacting their HR department directly.

In general, some common elements of parental leave policies include:

* Time off for new parents to bond with their child
* Compensation during the leave period (e.g., pay continuation)
* Eligibility and application processes

Here are a few examples of companies that offer secondary caregivers (also known as second-time parents) leave policies:

1. Facebook: Offers 26 weeks of parental leave, including bonding time for both primary and secondary caregivers.
2. Google: Provides 18 weeks of paid parental leave, with the option to

'According to [Source 1 | PARENTAL LEAVE POLICY], secondary caregivers receive 6 weeks of fully paid parental leave. (Policy section: "Secondary caregivers receive...")\n\nAnswering your question based on this HR policy document!'